# A2 — Double Machine Learning (PLR) sobre IHDP medium

**Curso:** Sistema de Inteligencia Causal · *profesionales*  
**Sesiones:** S8–S11 · **Módulo 2**  
**Lectura monografía:** Cap. **9** Features sin fuga · Cap. **10** Ortogonalidad · Cap. **11** DML · Cap. **14** Validación  
**Labs:** L08 · L09 · L10 · L11  
**Dataset:** `synthetic/inference/ihdp_medium`  
**Técnica:** PLR/DML, cross-fitting, Neyman, sensibilidad de *learners*  
**Alimenta Serie B:** **B1** (`artifacts/A2/dml_estimator.joblib` → API FastAPI)

### Pregunta de valor (disrupción)

> *Un buen predictor del outcome no dice cuánto **cambia** el outcome si intervenimos.*  
> *El ATE de DML responde: ¿cuánto mueve la intervención en promedio, con IC?*

### Objetivos de aprendizaje

1. Especificar el **PLR** y el estimando ATE.
2. Implementar **cross-fitting** sin fuga entre folds.
3. Verificar residuales de **ortogonalidad de Neyman**.
4. Comparar learners de nuisance (p. ej. Lasso vs boosting).
5. Reportar **ATE ± IC** y **serializar** el estimador oficial para B1/B2.
6. Interpretar sin usar ground truth en el cuerpo principal.


### Decisión de negocio

> *Si solo predijéramos el outcome, ordenaríamos unidades por su nivel esperado — sin decir si intervenir mueve la aguja.*  
> *Con el efecto (ATE de DML, con IC), respondemos la pregunta de negocio real: ¿vale la pena invertir en la intervención, en promedio, y con qué certeza?*

Esta es la **decisión de negocio** que alimenta el go/no-go de B1: el mismo ATE que aquí se estima y serializa es el que la API de producción va a responder.

## Aviso sobre los datos (transparencia)

Este notebook usa un escenario del **Causal Benchmark Ecosystem**.

Los archivos en `datasets/synthetic/inference/` **no son datos clínicos ni administrativos reales**.  
Lo auténtico de la literatura son **nombres de variables, diseño del estudio y órdenes de magnitud de referencia** (IHDP-style, efectos heterogéneos simulados).  
Los valores numéricos se generan con SEM / fórmulas documentadas para que exista **ground truth** (`truth_*.csv`) usable en evaluación y en el evaluación docente.

| Capa | Archivos | Quién la ve en este notebook |
|------|----------|------------------------------|
| Observables | `observed_*.csv` | **Siempre** (flujo del estudiante) |
| Ground truth | `truth_*.csv` (`y0`, `y1`, `tau`) | **Solo** sección final marcada «evaluación con truth» |
| Identificación / scorecard | `identification.json`, `scorecard.json` | Lectura guiada; no copiar ciegamente |

Consulta `metadata.json` (p. ej. `scenario_id`, `num_covariates`, `description`).  
**Política:** no uses `y0`, `y1` ni `tau` fuera de la sección final de evaluación.


## 1. Setup y carga del escenario

Semilla global del curso: `RANDOM_STATE = 42`.  
Splits oficiales del escenario (`observed_train/validation/test`); no re-particionamos.

Si `econml` / `doubleml` no están instalados, el notebook **degrada** a un PLR sklearn residual-on-residual con el mismo cross-fitting (documentado en celdas).


In [ ]:
# Paths relativos al monorepo (kernel-less: funciona con nbconvert/pytest)
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_NOTEBOOK_DIR = Path.cwd()
_CANDIDATES = [
    _NOTEBOOK_DIR.parent,  # .../03_notebooks_caso_estudio if cwd is notebooks/
    _NOTEBOOK_DIR,         # if cwd is already package root
]
for _p in _CANDIDATES:
    if _p is None:
        continue
    if (_p / "lib" / "paths.py").is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        PACKAGE_ROOT = _p
        break
else:
    here = Path.cwd().resolve()
    PACKAGE_ROOT = None
    for parent in [here, *here.parents]:
        if (parent / "lib" / "paths.py").is_file():
            sys.path.insert(0, str(parent))
            PACKAGE_ROOT = parent
            break
    if PACKAGE_ROOT is None:
        raise RuntimeError(
            "No se encontró lib/paths.py. Ejecuta desde "
            "03_notebooks_caso_estudio/ o notebooks/."
        )

from lib.paths import (  # noqa: E402
    RANDOM_STATE,
    SCENARIO_A2,
    SCENARIO_IDS,
    artifacts_dir,
    data_root,
    package_root,
)
from lib.data_io import (  # noqa: E402
    assert_no_truth_columns,
    load_scenario,
)
from lib.identification import (  # noqa: E402
    diff_in_means,
    fit_propensity,
    ols_ate,
)
from lib.dml_utils import (  # noqa: E402
    compare_learners_plr,
    cross_fit_residuals,
    default_model_t,
    default_model_y,
    export_a2_artifacts,
    fit_plr,
    fit_plr_sklearn,
    load_dml_artifact,
    neyman_orthogonality_checks,
    plr_from_residuals,
    save_dml_artifact,
    wrap_plr_result,
)
from lib.metrics import ate_absolute_error, summarize_ate_comparison  # noqa: E402

np.random.seed(RANDOM_STATE)
apply_style = True
try:
    from lib.plotting import apply_course_style
    apply_course_style()
except Exception:
    apply_style = False

print("PACKAGE_ROOT :", package_root())
print("DATA_ROOT    :", data_root())
print("RANDOM_STATE :", RANDOM_STATE)
print("SCENARIO_A2  :", SCENARIO_A2, "→", SCENARIO_IDS.get(SCENARIO_A2))

# Backend availability (graceful degradation)
_HAS_ECONML = False
_HAS_DOUBLEML = False
try:
    import econml  # noqa: F401
    _HAS_ECONML = True
except ImportError:
    pass
try:
    import doubleml  # noqa: F401
    _HAS_DOUBLEML = True
except ImportError:
    pass
BACKEND = "auto"  # tries econml, else sklearn residual-on-residual
print(f"econml={_HAS_ECONML}  doubleml={_HAS_DOUBLEML}  backend_default={BACKEND!r}")


In [ ]:
# Carga SOLO observables + sidecars (sin truth)
bundle = load_scenario(SCENARIO_A2)
assert_no_truth_columns(bundle.train, context="train")
assert_no_truth_columns(bundle.test, context="test")

meta = bundle.meta
ident = bundle.identification
scorecard = bundle.scorecard

print("scenario_id :", bundle.scenario_id)
print("shapes      : train", bundle.train.shape,
      "| val", bundle.validation.shape,
      "| test", bundle.test.shape)
print("description :", meta.get("description"))
print("num_covars  :", meta.get("num_covariates"))
print("treatment_r :", meta.get("treatment_ratio"))
print("generator seed (dataset):", meta.get("seed"),
      "| course RANDOM_STATE:", RANDOM_STATE)
# Nota: ate_true vive en metadata del generador; NO lo usamos para estimar.
print("metadata keys:", sorted(meta.keys()))


In [ ]:
# Roles de columnas y matrices de trabajo
Y_COL = bundle.outcome_column()    # outcome
T_COL = bundle.treatment_column()  # treatment
X_COLS = bundle.covariate_columns()

print(f"Y = {Y_COL!r}  |  T = {T_COL!r}")
print(f"X ({len(X_COLS)}): {X_COLS}")

Y_tr, T_tr, X_tr = bundle.ytx("train")
Y_va, T_va, X_va = bundle.ytx("validation")
Y_te, T_te, X_te = bundle.ytx("test")

print("train treatment rate:", float(T_tr.mean()))
print("train outcome mean  :", float(Y_tr.mean()))
assert list(X_tr.columns) == X_COLS
# Leakage guard already enforced via assert_no_truth_columns above


## 2. Modelo PLR e identificación (repaso)

### Especificación Partially Linear Regression (PLR)

\[
Y = \theta\, T + g(X) + \varepsilon, \qquad
T = m(X) + \eta
\]

- **Estimando:** ATE \(= \theta\) bajo el PLR (efecto constante / componente paramétrico del tratamiento).
- **Nuisance functions:** \(g(X) = \mathbb{E}[Y\mid X]\) (en la forma residualizada) y \(m(X) = \mathbb{E}[T\mid X]\).
- **DML / residual-on-residual:** con residuales de cross-fitting
  \(\tilde Y = Y - \hat g^{(-k)}(X)\), \(\tilde T = T - \hat m^{(-k)}(X)\),
  se estima \(\hat\theta\) por OLS de \(\tilde Y\) sobre \(\tilde T\) (sin intercepto).

### Supuestos (declarar, no «demostrar» con truth)

1. **Ignorabilidad condicional:** \(\{Y(0), Y(1)\} \perp T \mid X\).
2. **Overlap:** \(0 < \mathbb{P}(T=1\mid X) < 1\) casi seguro.
3. **SUTVA:** no interferencia; tratamiento bien definido.
4. **Especificación PLR** (o, en general, Neyman-ortogonalidad del score DML).

El archivo `identification.json` anota el diseño del SEM del benchmark; **no** es una prueba empírica de ignorabilidad en el mundo real.


In [ ]:
# Lectura guiada de identification + scorecard (piso del curso)
print("=== identification.json (claves) ===")
if ident:
    for k in list(ident.keys())[:12]:
        v = ident[k]
        if isinstance(v, (str, int, float, bool)) or v is None:
            print(f"  {k}: {v}")
        else:
            print(f"  {k}: {type(v).__name__}")
else:
    print("  (vacío o ausente)")

print("\n=== scorecard baselines (ATE estimates; NO son truth) ===")
br = (scorecard or {}).get("baseline_results", {})
for name, block in br.items():
    if isinstance(block, dict) and "ate_estimate" in block:
        print(f"  {name}: ate={block['ate_estimate']:.4f}",
              f"AE={block.get('absolute_error', float('nan')):.4f}")


## 3. Baselines: diferencia de medias, OLS e IPW

Antes de DML, recuperamos estimadores simples sobre **train** (mismo espíritu que A1, con 25 covariables).


In [ ]:
def ipw_ate(y, t, X, *, trim_low=0.02, trim_high=0.98, random_state=RANDOM_STATE):
    """IPW ATE with logistic propensity and clipping (student-safe; observed only)."""
    y_a = np.asarray(y, dtype=float).reshape(-1)
    t_a = np.asarray(t, dtype=float).reshape(-1)
    _, e = fit_propensity(X, t_a, random_state=random_state)
    e = np.clip(e, trim_low, trim_high)
    # Hájek-style stabilized IPW
    w1 = t_a / e
    w0 = (1.0 - t_a) / (1.0 - e)
    mu1 = np.sum(w1 * y_a) / np.sum(w1)
    mu0 = np.sum(w0 * y_a) / np.sum(w0)
    return float(mu1 - mu0), e

ate_dim = diff_in_means(Y_tr, T_tr)
ate_ols = ols_ate(Y_tr, T_tr, X_tr)
ate_ipw, e_tr = ipw_ate(Y_tr, T_tr, X_tr)

baseline_tbl = pd.DataFrame([
    {"estimator": "diff_in_means", "ate_hat": ate_dim, "split": "train"},
    {"estimator": "ols_adjusted", "ate_hat": ate_ols, "split": "train"},
    {"estimator": "ipw_clip_0.02_0.98", "ate_hat": ate_ipw, "split": "train"},
])
print(baseline_tbl.to_string(index=False))
print(f"\npropensity train: min={e_tr.min():.3f} max={e_tr.max():.3f} mean={e_tr.mean():.3f}")


## 4. DML-PLR con cross-fitting (K=5)

### Cross-fitting sin fuga

En cada fold \(k\):

1. Entrenar \(\hat g^{(-k)}\) y \(\hat m^{(-k)}\) **solo** en folds \(\neq k\).
2. Predecir residuales **solo** en el fold \(k\).
3. Concatenar residuales OOF y estimar \(\theta\) en la muestra residualizada completa.

Implementación de referencia: `lib.dml_utils.cross_fit_residuals` + `plr_from_residuals`  
(y, si está disponible, `econml.dml.LinearDML`).

**Estimador oficial del curso para serialización B1:** PLR + **LassoCV** en nuisances (`model_y` / `model_t`), `n_folds=5`, `random_state=42`.


In [ ]:
N_FOLDS = 5

# --- Especificación 1: Lasso (oficial del curso) ---
est_lasso, res_lasso = fit_plr(
    Y_tr, T_tr, X_tr,
    backend=BACKEND,
    model_y_kind="lasso",
    model_t_kind="lasso",
    n_folds=N_FOLDS,
    random_state=RANDOM_STATE,
)

print("=== PLR Lasso ===")
print(json.dumps(res_lasso.to_dict(), indent=2, default=str))
if est_lasso is not None:
    print("underlying estimator:", type(est_lasso).__name__)
else:
    print("backend efectivo: sklearn residual-on-residual (econml no disponible o no usado)")


## 5. Diagnóstico de ortogonalidad de Neyman

Con residuales de cross-fitting \(\tilde Y, \tilde T\) y predicciones OOF \(\hat g(X), \hat m(X)\):

- \(\mathrm{corr}(\tilde Y, \hat g)\) y \(\mathrm{corr}(\tilde T, \hat m)\) deberían ser **bajas** (proyección ortogonal aproximada).
- \(\mathrm{mean}(\tilde Y)\), \(\mathrm{mean}(\tilde T)\) cercanas a 0 si los nuisances están bien calibrados en media.
- El scatter \(\tilde Y\) vs \(\tilde T\) visualiza la pendiente \(\hat\theta\).


In [ ]:
# Residuales OOF explícitos (misma especificación Lasso) para diagnóstico
my = default_model_y("lasso", random_state=RANDOM_STATE)
mt = default_model_t("lasso", random_state=RANDOM_STATE)
y_res, t_res, y_hat, t_hat = cross_fit_residuals(
    Y_tr, T_tr, X_tr, my, mt, n_folds=N_FOLDS, random_state=RANDOM_STATE
)
# Consistencia con plr_from_residuals
res_from_resid = plr_from_residuals(y_res, t_res)
print("theta from residuals:", res_from_resid.ate, "se:", res_from_resid.se)

neyman = neyman_orthogonality_checks(y_res, t_res, y_hat, t_hat)
print("\n=== Neyman residual diagnostics ===")
for k, v in neyman.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(t_res, y_res, s=8, alpha=0.35, c="#1f4e79")
# línea de PLR
tt = np.linspace(t_res.min(), t_res.max(), 100)
axes[0].plot(tt, res_from_resid.ate * tt, color="#c0392b", lw=2, label=f"θ̂={res_from_resid.ate:.3f}")
axes[0].set_xlabel(r"$	ilde{T}$ (residual treatment)")
axes[0].set_ylabel(r"$	ilde{Y}$ (residual outcome)")
axes[0].set_title("PLR residual-on-residual")
axes[0].legend()

axes[1].hist(y_res, bins=40, alpha=0.55, label=r"$	ilde{Y}$", color="#1f4e79")
axes[1].hist(t_res, bins=40, alpha=0.55, label=r"$	ilde{T}$", color="#c0392b")
axes[1].set_title("Distribución de residuales OOF")
axes[1].legend()
plt.tight_layout()
plt.show()

# Sanidad metodológica (heurística pedagógica, no prueba formal)
print("\nHeurística: |corr(y_res, y_hat)| y |corr(t_res, t_hat)| pequeños es deseable.")
print(f"  |corr_y_res_y_hat| = {abs(neyman.get('corr_y_res_y_hat', float('nan'))):.4f}")
print(f"  |corr_t_res_t_hat| = {abs(neyman.get('corr_t_res_t_hat', float('nan'))):.4f}")


## 6. Sensibilidad: Lasso vs HistGradientBoosting

Repetimos el PLR con *learners* flexibles en los nuisances.  
La **estabilidad** de \(\hat\theta\) entre especificaciones es más informativa que perseguir un único número.


In [ ]:
# Comparación de learners (sklearn backend para control total del cross-fit)
learner_table = compare_learners_plr(
    Y_tr, T_tr, X_tr,
    learner_specs=(
        ("lasso", "lasso", "lasso"),
        ("hist_gb", "hgb", "hgb"),
    ),
    n_folds=N_FOLDS,
    random_state=RANDOM_STATE,
    backend="sklearn",
)

# También la corrida principal (BACKEND auto) ya hecha con Lasso
learner_table = learner_table.copy()
print(learner_table.to_string(index=False))

# Intervalos
fig, ax = plt.subplots(figsize=(7, 3.5))
ypos = np.arange(len(learner_table))
ax.errorbar(
    learner_table["ate"],
    ypos,
    xerr=1.96 * learner_table["se"].astype(float),
    fmt="o",
    color="#1f4e79",
    capsize=4,
)
ax.set_yticks(ypos)
ax.set_yticklabels(learner_table["learner"])
ax.axvline(res_lasso.ate, color="#c0392b", ls="--", label="oficial (Lasso / backend auto)")
ax.set_xlabel("ATE (PLR)")
ax.set_title("Sensibilidad de learners (IC ~95%)")
ax.legend(loc="best")
plt.tight_layout()
plt.show()


## 7. Tabla comparativa de especificaciones

Combinamos baselines + DML. **Sin** columna de error vs truth aquí (eso es § evaluación final).


In [ ]:
rows = [
    {"estimator": "diff_in_means", "ate": ate_dim, "se": np.nan, "ci_low": np.nan, "ci_high": np.nan},
    {"estimator": "ols_adjusted", "ate": ate_ols, "se": np.nan, "ci_low": np.nan, "ci_high": np.nan},
    {"estimator": "ipw_clip", "ate": ate_ipw, "se": np.nan, "ci_low": np.nan, "ci_high": np.nan},
    {
        "estimator": f"dml_plr_lasso[{res_lasso.method}]",
        "ate": res_lasso.ate,
        "se": res_lasso.se,
        "ci_low": res_lasso.ci_low,
        "ci_high": res_lasso.ci_high,
    },
]
for _, r in learner_table.iterrows():
    rows.append({
        "estimator": f"dml_plr_{r['learner']}_sklearn",
        "ate": r["ate"],
        "se": r["se"],
        "ci_low": r["ci_low"],
        "ci_high": r["ci_high"],
    })

comparison = pd.DataFrame(rows).drop_duplicates(subset=["estimator"], keep="first")
print(comparison.to_string(index=False))

# Scorecard como piso (no como verdad)
print("\nPiso del curso (scorecard, AE vs generador — informativo):")
for name in ("inference_naive_test_difference", "inference_ols_adjusted", "inference_ipw_clipped_0.02_0.98"):
    block = br.get(name, {})
    if block:
        print(f"  {name}: ate_estimate={block.get('ate_estimate'):.4f} AE={block.get('absolute_error'):.4f}")


## Refutación por permutación (placebo)

Misma prueba que en A3: permutamos el tratamiento y reestimamos el CATE. Si el procedimiento es válido, el ATE permutado debe ser indistinguible de cero — el *shuffle* destruye la relación tratamiento→outcome sin tocar covariables. Usamos una sola permutación (`n_perm=1`), como el evaluación docente, para respetar el presupuesto de tiempo del curso. No requiere truth: la prueba se apoya solo en observables.

In [ ]:
# Placebo por permutación del tratamiento (n_perm=1, como el evaluación docente de A3)
from lib.cate_utils import placebo_permute_treatment

placebo = placebo_permute_treatment(
    Y_tr, T_tr, X_tr,
    learner="t",
    n_perm=1,
    random_state=RANDOM_STATE,
)

print("=== Refutación por permutación (train, sin truth) ===")
print(f"ATE placebo (T permutado) : {placebo['ate_placebo']:.4f}")
print(f"mean|tau_hat| placebo     : {placebo['mean_abs_tau']:.4f}")
print(f"learner / n_perm          : {placebo['learner']} / {placebo['n_perm']}")

# Umbral orientativo del curso (mismo que el evaluación docente de A3): |ATE_placebo| < 0.5
UMBRAL_PLACEBO = 0.5
ok_placebo = abs(placebo["ate_placebo"]) < UMBRAL_PLACEBO
print(f"|ATE_placebo| < {UMBRAL_PLACEBO}  →  {'OK' if ok_placebo else 'REVISAR'}")

## 8. Serialización del estimador oficial → Serie B (B1)

Artefactos esperados:

```text
artifacts/A2/dml_estimator.joblib
artifacts/A2/feature_list.json
artifacts/A2/training_meta.json
```

Contrato B1: el objeto joblib expone al menos `const_marginal_ate()` (y `predict(X)` con ATE constante).  
Alternativa kernel-less: `python scripts/export_a2_artifacts.py`.


In [ ]:
# Estimador oficial del curso: PLR + Lasso, ya ajustado en res_lasso
artifact_dir = artifacts_dir("A2")
artifact_dir.mkdir(parents=True, exist_ok=True)

payload = wrap_plr_result(res_lasso, feature_list=X_COLS, underlying=est_lasso)
if est_lasso is not None:
    payload.extra_["underlying_type"] = type(est_lasso).__name__

training_meta = {
    "notebook_id": "A2",
    "scenario": SCENARIO_A2,
    "scenario_id": bundle.scenario_id,
    "seed": RANDOM_STATE,
    "dataset_generator_seed": meta.get("seed"),
    "split_used": "train",
    "n": int(res_lasso.n),
    "n_folds": int(res_lasso.n_folds or N_FOLDS),
    "backend_method": res_lasso.method,
    "model_y_kind": "lasso",
    "model_t_kind": "lasso",
    "feature_list": list(X_COLS),
    "ate_point_estimate": float(res_lasso.ate),
    "se": None if res_lasso.se is None else float(res_lasso.se),
    "ci_low": None if res_lasso.ci_low is None else float(res_lasso.ci_low),
    "ci_high": None if res_lasso.ci_high is None else float(res_lasso.ci_high),
    "ci_level": 0.95,
    "estimand": "ATE",
    "model": "PLR",
    "assumptions": [
        "conditional_ignorability",
        "overlap",
        "SUTVA",
        "PLR_specification",
    ],
}

joblib_path = artifact_dir / "dml_estimator.joblib"
save_dml_artifact(
    payload,
    joblib_path,
    feature_list=X_COLS,
    training_meta=training_meta,
)

# Verificación de carga (contrato B1)
loaded = load_dml_artifact(joblib_path)
ate_loaded = float(loaded.const_marginal_ate())
pred_head = loaded.predict(X_te.head(3))
print("Wrote:", joblib_path)
print("       ", artifact_dir / "feature_list.json")
print("       ", artifact_dir / "training_meta.json")
print(f"const_marginal_ate() = {ate_loaded:.6f}")
print("predict(X_test[:3]) =", pred_head)

with (artifact_dir / "feature_list.json").open() as f:
    fl = json.load(f)
assert fl == list(X_COLS), "feature_list.json must match training covariates"
print("feature_list OK:", len(fl), "covariates")


## 9. Nota tipo model card (anticipa B2)

Completa brevemente (el evaluación docente comprueba longitud mínima de texto metodológico):

### Plantilla de model card mínima

| Campo | Contenido |
|-------|-----------|
| **Estimando** | ATE bajo PLR (θ) |
| **Población** | Unidades del escenario `synthetic_inference_ihdp_medium` (n train = 1200) |
| **Tratamiento / outcome** | `treatment` / `outcome` |
| **Controles** | `x0`…`x24` (25) |
| **Método** | DML-PLR, K=5, nuisances LassoCV (oficial) |
| **Supuestos** | Ignorabilidad condicional, overlap, SUTVA, PLR |
| **Límites** | Datos sintéticos IHDP-style; homogeneidad del PLR; sensibilidad a learners |
| **Artefacto** | `artifacts/A2/dml_estimator.joblib` → B1 |

### Reflexión (escribir aquí)

- ¿Por qué el cross-fitting evita sesgo de sobreajuste de los nuisances en la misma muestra?
- ¿Qué cambió al pasar de Lasso a HistGradientBoosting? ¿El IC se solapa?
- ¿Qué riesgo de validez externa tendría un despliegue B1 de este joblib?


### Respuesta de referencia (modelo de escritura — adaptar con tus números)

El estimando es el **ATE** del tratamiento binario sobre `outcome` en la población del escenario sintético IHDP medium, identificado bajo ignorabilidad condicional en \(X=(x_0,\ldots,x_{24})\), overlap y SUTVA. Ajustamos un **PLR** vía DML: residualizamos \(Y\) y \(T\) con modelos de nuisance entrenados por **cross-fitting K=5** (sin reutilizar el fold de evaluación al entrenar \(g\) y \(m\)), y estimamos \(\theta\) por regresión residual-on-residual con error estándar robusto e intervalo de confianza al 95 %. La especificación oficial del curso usa **LassoCV** en ambos nuisances por estabilidad y parsimonia en p=25; como sensibilidad comparamos **HistGradientBoosting**. Los residuales de Neyman deben mostrar baja correlación con las predicciones OOF de los nuisances. Los datos son **sintéticos** (SEM documentado): el joblib sirve para el pipeline de producción B1, no como evidencia clínica. Un riesgo de validez externa es que el PLR impone efecto homogéneo y que el solapamiento / la ley de asignación pueden diferir fuera del simulador.


## Sección de evaluación reservada

La edición pública termina antes de las celdas que requieren `truth`, `y0`, `y1` o `tau` simulados. La evaluación docente se ejecuta en un entorno separado. Continúa la interpretación con datos observados, incertidumbre y supuestos declarados.
